In [26]:
import re, time
import numpy as np, pandas as pd
import sys
print(sys.executable)
%pip install tensorflow
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ---- Course-style model comparison, part 1: baselines, linear regression, ridge (GroupKFold CV).
# Only the train and validation splits are used here; the test split stays untouched.
T = pd.read_parquet("Amazon/model_table.parquet")

PACK = re.compile(r"(?:pack of|set of|box of|case of)\s*(\d+)|(\d+)\s*[- ]?(?:pack|pk|count|ct|pcs|pieces|piece|pc|sets)\b", re.I)
def pack_count(title):
    m = PACK.search(title) if isinstance(title, str) else None
    if not m:
        return 1
    n = int(m.group(1) or m.group(2))
    return n if 1 <= n <= 1000 else 1

def make_sets(g_, ratio=1.5, k=4):
    g_ = g_.sort_values("price")
    idx, p = g_.index.tolist(), g_["price"].tolist()
    out, i = [], 0
    while i + k <= len(p):
        if p[i + k - 1] / p[i] <= ratio:
            out.append(idx[i:i + k]); i += k
        else:
            i += 1
    return out

T["pack"] = T["title"].map(pack_count)
MANUAL_EXCLUDE = r"Executive Notebook|Pins & Tacks|Pen Refill|Planner"   # a priori judgment, made before looking at ratings
tr, va = T[T.split == "train"], T[T.split == "val"]
y_tr, y_va = tr["mean"].values, va["mean"].values
groups_tr = tr["family"].values
ybar = y_tr.mean()

def osr2(y, p, base):
    return 1 - ((y - p) ** 2).sum() / ((y - base) ** 2).sum()

# Features (all preprocessing is fitted on the training split only)
txt = lambda d: d["title"] + " " + d["features"] + " " + d["description"]
tfidf = TfidfVectorizer(min_df=3, ngram_range=(1, 2), max_features=50000, sublinear_tf=True).fit(txt(tr))
ohe = OneHotEncoder(handle_unknown="ignore").fit(tr[["cat_leaf"]].fillna("NA"))
med = tr["price"].median()
def num(d):
    return np.c_[np.log1p(d["price"].fillna(med)), d["price"].isna().astype(float),
                 np.log1p(d["title"].str.len()), np.log1p(d["features"].str.len()), np.log1p(d["description"].str.len())]
sc = StandardScaler().fit(num(tr))
def struct(d):
    return hstack([ohe.transform(d[["cat_leaf"]].fillna("NA")), csr_matrix(sc.transform(num(d)))]).tocsr()

Xs_tr, Xs_va = struct(tr), struct(va)
Xt_tr, Xt_va = tfidf.transform(txt(tr)), tfidf.transform(txt(va))
Xst_tr, Xst_va = hstack([Xs_tr, Xt_tr]).tocsr(), hstack([Xs_va, Xt_va]).tocsr()
E = np.load("Amazon/emb_minilm_raw.npy")
assert len(E) == len(T)
E_tr, E_va = E[tr.index.values], E[va.index.values]
XsE_tr, XsE_va = hstack([Xs_tr, csr_matrix(E_tr)]).tocsr(), hstack([Xs_va, csr_matrix(E_va)]).tocsr()

# Validation sets: built inside the val split only, from category, pack size and price (no ratings used)
ok = va[va["set_eligible"] & ~va["cat_leaf"].fillna("").str.contains(MANUAL_EXCLUDE, case=False, regex=True)]
sets_va = []
for (_, _), g_ in ok.groupby(["cat_leaf", "pack"]):
    sets_va += make_sets(g_)
pos = {ix: k for k, ix in enumerate(va.index)}
sets_pos = np.array([[pos[i] for i in s] for s in sets_va])
truth_sets = y_va[sets_pos]
R, O = truth_sets.mean(1), truth_sets.max(1)
print(f"Validation sets: {len(sets_va)} | headroom O-R: {np.mean(O - R):.3f} stars")

tie = np.random.default_rng(7).normal(0, 1e-9, len(va))   # fixed-seed tie breaking
def boot_ci(x, B=2000, seed=0):
    r = np.random.default_rng(seed); x = np.asarray(x)
    return np.percentile(x[r.integers(0, len(x), (B, len(x)))].mean(1), [5, 95])

results = []
def add_result(name, p_tr, p_va, tuned=""):
    j = (p_va + tie)[sets_pos].argmax(1)
    M = truth_sets[np.arange(len(j)), j]
    lo, hi = boot_ci(M - R)
    row = {"model": name, "tuned": tuned,
           "train R2": osr2(y_tr, p_tr, ybar), "val OSR2": osr2(y_va, p_va, ybar),
           "val MAE": mean_absolute_error(y_va, p_va),
           "val RMSE": mean_squared_error(y_va, p_va) ** 0.5,
           "M-R": (M - R).mean(), "90% CI": f"[{lo:.3f}, {hi:.3f}]", "top1": (j == truth_sets.argmax(1)).mean()}
    results[:] = [r for r in results if r["model"] != name] + [row]   # re-running a cell replaces its old row

cv = GroupKFold(n_splits=3)
def tune_ridge(name, Xa, Xb):
    t0 = time.time()
    gs = GridSearchCV(Ridge(), {"alpha": [0.03, 0.1, 0.3, 1, 3, 10, 30, 100]}, cv=cv, scoring="neg_root_mean_squared_error")
    gs.fit(Xa, y_tr, groups=groups_tr)           # refit=True: best alpha is refit on the full training split
    add_result(name, gs.predict(Xa), gs.predict(Xb), f"alpha={gs.best_params_['alpha']}")
    print(f"{name}: done in {time.time() - t0:.0f}s")

# 1) Baselines
add_result("Baseline: train mean", np.full(len(tr), ybar), np.full(len(va), ybar))
lm = tr.groupby("cat_leaf")["mean"].mean()
add_result("Baseline: leaf mean", tr["cat_leaf"].map(lm).fillna(ybar).values, va["cat_leaf"].map(lm).fillna(ybar).values)

# 2) Linear regression and ridge (lambda chosen by GroupKFold CV on the training split)
ols = LinearRegression().fit(Xs_tr, y_tr)
add_result("Linear regression (structured)", ols.predict(Xs_tr), ols.predict(Xs_va))
tune_ridge("Ridge (structured)", Xs_tr, Xs_va)
tune_ridge("Ridge (structured + TF-IDF)", Xst_tr, Xst_va)
tune_ridge("Ridge (structured + MiniLM)", XsE_tr, XsE_va)

print(pd.DataFrame(results).round(3).to_string(index=False))

/Users/thea/Documents/Harvard/Fall/15.071 ai-edge/Final/.venv/bin/python
  Using cached tensorflow-2.21.0-cp313-cp313-macosx_12_0_arm64.whl.metadata (4.4 kB)
  Using cached absl_py-2.5.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached gast-0.7.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-1-py2.py3-none-macosx_11_0_arm64.whl.metadata (5.2 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached protobuf-7.36.2-cp310-abi3-macosx_10_9_universal2.whl.metadata (595 bytes)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached six-1.17.0-py2.py3-none-any.whl.metadata (1.7 kB)
  Using cached termcolor-3.3.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached grpcio-1.85.1-cp313-cp313-macosx_11_0_univer

In [27]:
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

# ---- Course-style model comparison, part 2: tree ensembles (GroupKFold CV on the training split).
# Trees get dense inputs: leaf-category dummies (frequent leaves only), standardized numeric features, and a text block.
svd = TruncatedSVD(n_components=100, random_state=0).fit(Xt_tr)
S_tr, S_va = svd.transform(Xt_tr), svd.transform(Xt_va)

freq = tr["cat_leaf"].value_counts()
leaves = freq[freq >= 50].index
def leaf_dummies(d):
    c = d["cat_leaf"].where(d["cat_leaf"].isin(leaves))
    return pd.get_dummies(pd.Categorical(c, categories=leaves)).values.astype(float)

N_tr, N_va = sc.transform(num(tr)), sc.transform(num(va))
D_svd_tr, D_svd_va = np.hstack([leaf_dummies(tr), N_tr, S_tr]), np.hstack([leaf_dummies(va), N_va, S_va])
D_emb_tr, D_emb_va = np.hstack([leaf_dummies(tr), N_tr, E_tr]), np.hstack([leaf_dummies(va), N_va, E_va])
print("Tree inputs:", D_svd_tr.shape, D_emb_tr.shape)

def tune_tree(name, estimator, grid, Xa, Xb):
    t0 = time.time()
    gs = GridSearchCV(estimator, grid, cv=cv, scoring="neg_root_mean_squared_error", n_jobs=-1)
    gs.fit(Xa, y_tr, groups=groups_tr)
    add_result(name, gs.predict(Xa), gs.predict(Xb), str(gs.best_params_))
    print(f"{name}: done in {time.time() - t0:.0f}s")

rf = RandomForestRegressor(n_estimators=200, random_state=0, n_jobs=-1)
tune_tree("Random forest (structured + TF-IDF SVD)", rf,
          {"max_features": [0.3, 0.6], "min_samples_leaf": [5, 20]}, D_svd_tr, D_svd_va)

hgb_grid = {"learning_rate": [0.05, 0.1], "max_leaf_nodes": [8, 31], "max_iter": [100, 300]}
hgb = HistGradientBoostingRegressor(early_stopping=False, random_state=0)
tune_tree("Gradient boosting (structured + TF-IDF SVD)", hgb, hgb_grid, D_svd_tr, D_svd_va)
tune_tree("Gradient boosting (structured + MiniLM)", hgb, hgb_grid, D_emb_tr, D_emb_va)

print(pd.DataFrame(results).sort_values("val OSR2", ascending=False).round(3).to_string(index=False))

Tree inputs: (12934, 172) (12934, 456)
Random forest (structured + TF-IDF SVD): done in 122s
Gradient boosting (structured + TF-IDF SVD): done in 33s
Gradient boosting (structured + MiniLM): done in 73s
                                      model                                                          tuned  train R2  val OSR2  val MAE  val RMSE    M-R          90% CI  top1
                Ridge (structured + TF-IDF)                                                        alpha=1     0.780     0.431    0.281     0.379  0.177  [0.140, 0.213] 0.473
Gradient boosting (structured + TF-IDF SVD) {'learning_rate': 0.05, 'max_iter': 300, 'max_leaf_nodes': 31}     0.729     0.411    0.287     0.385  0.155  [0.116, 0.199] 0.439
    Random forest (structured + TF-IDF SVD)                   {'max_features': 0.6, 'min_samples_leaf': 5}     0.806     0.406    0.287     0.387  0.132  [0.091, 0.176] 0.439
    Gradient boosting (structured + MiniLM)  {'learning_rate': 0.1, 'max_iter': 300, 'max_leaf_no

In [28]:
import os, re
import numpy as np, pandas as pd

MASK_BRAND = False      # False = raw text; True = brand-masked version (closer to the 4435 materials)
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
TAG = "masked" if MASK_BRAND else "raw"
recompute = False       # set True only if you want to re-encode
path = f"Amazon/emb_minilm_{TAG}.npy"

if recompute or not os.path.exists(path):
    import torch
    from sentence_transformers import SentenceTransformer

    def mask_store(text, store):
        s = (store or "").strip()
        return re.sub(re.escape(s), " ", text, flags=re.I) if len(s) >= 3 else text

    texts = (T["title"] + ". " + T["features"] + " " + T["description"]).str.slice(0, 1500)
    if MASK_BRAND:
        texts = pd.Series([mask_store(t, s) for t, s in zip(texts, T["store"])], index=T.index)

    device = "mps" if torch.backends.mps.is_available() else "cpu"   # if mps raises an error, set device = "cpu"
    print("device:", device)
    st = SentenceTransformer(MODEL_NAME, device=device)
    st.max_seq_length = 256
    E = st.encode(texts.tolist(), batch_size=64, show_progress_bar=True, normalize_embeddings=True)
    np.save(path, E.astype("float32"))
else:
    E = np.load(path)
print("Embeddings:", E.shape)

Embeddings: (21462, 384)


In [29]:
# ---- What drives the text signal? Text-field ablations and brand masking (ridge, alpha fixed at the CV choice).
ALPHA = 1   # the value selected by GroupKFold CV for "Ridge (structured + TF-IDF)"; change it if your table shows another value

def mask_store(text, store):
    s = (store or "").strip()
    return re.sub(re.escape(s), " ", text, flags=re.I) if len(s) >= 3 else text

def fit_variant(name, make_text):
    vec = TfidfVectorizer(min_df=3, ngram_range=(1, 2), max_features=50000, sublinear_tf=True).fit(make_text(tr))
    A_tr = hstack([Xs_tr, vec.transform(make_text(tr))]).tocsr()
    A_va = hstack([Xs_va, vec.transform(make_text(va))]).tocsr()
    m = Ridge(alpha=ALPHA).fit(A_tr, y_tr)
    add_result(name, m.predict(A_tr), m.predict(A_va), f"alpha={ALPHA}")
    return vec, m

masked = lambda d: pd.Series([mask_store(t, s) for t, s in zip(txt(d), d["store"])], index=d.index)

vec_all, m_all = fit_variant("Ridge: all text (raw)", txt)
fit_variant("Ridge: title only", lambda d: d["title"])
fit_variant("Ridge: features only", lambda d: d["features"])
fit_variant("Ridge: description only", lambda d: d["description"])
fit_variant("Ridge: all text, brand masked", masked)
print(pd.DataFrame([r for r in results if r["model"].startswith("Ridge: ")]).round(3).to_string(index=False))

# Top associations in the raw all-text model (associations with higher/lower mean ratings, not causal effects)
names = vec_all.get_feature_names_out()
coef = m_all.coef_[Xs_tr.shape[1]:]
order = np.argsort(coef)
print("\nTerms most associated with LOWER ratings:", ", ".join(names[order[:25]]))
print("\nTerms most associated with HIGHER ratings:", ", ".join(names[order[::-1][:25]]))

                        model   tuned  train R2  val OSR2  val MAE  val RMSE   M-R         90% CI  top1
        Ridge: all text (raw) alpha=1     0.780     0.431    0.281     0.379 0.177 [0.140, 0.213] 0.473
            Ridge: title only alpha=1     0.736     0.414    0.286     0.384 0.152 [0.112, 0.193] 0.426
         Ridge: features only alpha=1     0.740     0.381    0.294     0.395 0.168 [0.133, 0.205] 0.486
      Ridge: description only alpha=1     0.581     0.351    0.305     0.404 0.089 [0.049, 0.131] 0.331
Ridge: all text, brand masked alpha=1     0.778     0.428    0.282     0.380 0.173 [0.137, 0.211] 0.459

Terms most associated with LOWER ratings: 72, erase surface, stores, hp, osir, year guarantee, like pencil, wireless, remanufactured, shopping, bazic, dell, holiday greeting, home, with marker, self stick, board, sheets pad, one year, replacement for, shopping cart, erases like, printronic, applicable, all in

Terms most associated with HIGHER ratings: post it, pilot, cano

In [30]:
from sklearn.preprocessing import OneHotEncoder

# ---- Diagnostics: (A) does knowing the store/brand alone predict within-set winners? (B) text + store, (C) train on within-leaf deviations.
s_tr = tr["store"].str.lower().str.strip()
cnt = s_tr.value_counts()
keep_st = set(cnt[cnt >= 3].index)                      # stores with at least 3 training products; the rest -> "other"
def store_cat(d):
    s = d["store"].str.lower().str.strip()
    return s.where(s.isin(keep_st), "other").to_frame("store_c")
ohe_st = OneHotEncoder(handle_unknown="ignore").fit(store_cat(tr))
Xb_tr, Xb_va = ohe_st.transform(store_cat(tr)), ohe_st.transform(store_cat(va))
print(f"Stores kept as separate categories: {len(keep_st)} | share of train products in 'other': {(~s_tr.isin(keep_st)).mean():.2f}")

tune_ridge("Ridge (structured + store)", hstack([Xs_tr, Xb_tr]).tocsr(), hstack([Xs_va, Xb_va]).tocsr())
tune_ridge("Ridge (structured + TF-IDF + store)", hstack([Xst_tr, Xb_tr]).tocsr(), hstack([Xst_va, Xb_va]).tocsr())

# (C) Fit on deviations from the leaf mean: selection happens inside same-leaf sets, so the leaf offset is irrelevant there
off_tr = tr["cat_leaf"].map(lm).fillna(ybar).values
off_va = va["cat_leaf"].map(lm).fillna(ybar).values
gs = GridSearchCV(Ridge(), {"alpha": [0.03, 0.1, 0.3, 1, 3, 10, 30, 100]}, cv=cv, scoring="neg_root_mean_squared_error")
gs.fit(Xst_tr, y_tr - off_tr, groups=groups_tr)
add_result("Ridge (structured + TF-IDF), leaf-demeaned target", off_tr + gs.predict(Xst_tr), off_va + gs.predict(Xst_va),
           f"alpha={gs.best_params_['alpha']}")

keys = ["Ridge (structured + TF-IDF)", "Ridge (structured + store)", "Ridge (structured + TF-IDF + store)",
        "Ridge (structured + TF-IDF), leaf-demeaned target"]
print(pd.DataFrame([r for r in results if r["model"] in keys]).round(3).to_string(index=False))

# Within-leaf associations (from the leaf-demeaned model): associations, not causal effects
names = tfidf.get_feature_names_out()
coef = gs.best_estimator_.coef_[Xs_tr.shape[1]:]
order = np.argsort(coef)
print("\nWithin-leaf: terms associated with LOWER ratings:", ", ".join(names[order[:25]]))
print("\nWithin-leaf: terms associated with HIGHER ratings:", ", ".join(names[order[::-1][:25]]))

Stores kept as separate categories: 883 | share of train products in 'other': 0.32
Ridge (structured + store): done in 0s
Ridge (structured + TF-IDF + store): done in 21s
                                            model   tuned  train R2  val OSR2  val MAE  val RMSE   M-R         90% CI  top1
                      Ridge (structured + TF-IDF) alpha=1     0.780     0.431    0.281     0.379 0.177 [0.140, 0.213] 0.473
                       Ridge (structured + store) alpha=3     0.431     0.356    0.303     0.403 0.138 [0.098, 0.178] 0.412
              Ridge (structured + TF-IDF + store) alpha=3     0.672     0.446    0.277     0.374 0.170 [0.134, 0.210] 0.432
Ridge (structured + TF-IDF), leaf-demeaned target alpha=1     0.780     0.417    0.282     0.383 0.178 [0.143, 0.213] 0.473

Within-leaf: terms associated with LOWER ratings: 72, erase surface, like pencil, stores, osir, shopping, year guarantee, bazic, dell, remanufactured, holiday greeting, hp, self stick, sheets pad, erases like

In [31]:
# ---- Within-brand test: do the text models rank products correctly when both products come from the SAME store?
# Pairs: same store, same leaf category, same pack size, price ratio <= 1.5, different product families.
# Store identity cancels inside a pair, so any ranking skill has to come from the product-specific text and attributes.
elig = va[va["set_eligible"] & ~va["cat_leaf"].fillna("").str.contains(MANUAL_EXCLUDE, case=False, regex=True)].copy()
elig["store_k"] = elig["store"].str.lower().str.strip()
elig = elig[elig["store_k"] != ""]
pos_va = {ix: k for k, ix in enumerate(va.index)}

pairs = []
for (st_, leaf_, pk_), g_ in elig.groupby(["store_k", "cat_leaf", "pack"]):
    if len(g_) < 2:
        continue
    ids, pr, fam = g_.index.values, g_["price"].values, g_["family"].values
    for a in range(len(ids)):
        for b in range(a + 1, len(ids)):
            if fam[a] != fam[b] and max(pr[a], pr[b]) / min(pr[a], pr[b]) <= 1.5:
                pairs.append((st_, pos_va[ids[a]], pos_va[ids[b]]))
pair_df = pd.DataFrame(pairs, columns=["store", "i", "j"])
d_true = y_va[pair_df["j"].values] - y_va[pair_df["i"].values]
keep_p = d_true != 0
pair_df, d_true = pair_df[keep_p].reset_index(drop=True), d_true[keep_p]
print(f"Within-store pairs: {len(pair_df)} from {pair_df['store'].nunique()} stores")

def pair_acc(pred, mask=None, B=2000, seed=0):
    """Share of pairs ordered correctly; 90% interval from a bootstrap over stores."""
    m = np.ones(len(pair_df), bool) if mask is None else mask
    p = pred + tie
    ok_ = (np.sign(p[pair_df["j"].values] - p[pair_df["i"].values]) == np.sign(d_true))[m]
    per = pd.DataFrame({"store": pair_df["store"].values[m], "ok": ok_}).groupby("store")["ok"].agg(["sum", "count"])
    s_, c_ = per["sum"].values, per["count"].values
    idx = np.random.default_rng(seed).integers(0, len(per), (B, len(per)))
    boots = s_[idx].sum(1) / c_[idx].sum(1)
    return ok_.mean(), np.percentile(boots, [5, 95]), int(m.sum())

def refit(Xa, Xb, alpha):
    return Ridge(alpha=alpha).fit(Xa, y_tr).predict(Xb)

preds = {
    "Leaf mean (control: no skill expected)": va["cat_leaf"].map(lm).fillna(ybar).values,
    "Ridge (structured)": refit(Xs_tr, Xs_va, 3),
    "Ridge (structured + TF-IDF)": refit(Xst_tr, Xst_va, 1),
    "Ridge (structured + MiniLM)": refit(XsE_tr, XsE_va, 3),
}
big = np.abs(d_true) >= 0.3          # pairs with a clearly different true rating
rows_p = []
for name, p in preds.items():
    a1, ci1, n1 = pair_acc(p)
    a2, ci2, n2 = pair_acc(p, big)
    rows_p.append({"model": name, "pair acc (all)": a1, "90% CI": f"[{ci1[0]:.3f}, {ci1[1]:.3f}]",
                   f"acc (|diff|>=0.3, n={n2})": a2, "90% CI ": f"[{ci2[0]:.3f}, {ci2[1]:.3f}]"})
print(pd.DataFrame(rows_p).round(3).to_string(index=False))
print("Chance level is 0.5.")

Within-store pairs: 227 from 83 stores
                                 model  pair acc (all)         90% CI  acc (|diff|>=0.3, n=78)        90% CI 
Leaf mean (control: no skill expected)           0.493 [0.435, 0.551]                    0.513 [0.413, 0.604]
                    Ridge (structured)           0.520 [0.450, 0.583]                    0.513 [0.357, 0.631]
           Ridge (structured + TF-IDF)           0.595 [0.533, 0.653]                    0.667 [0.569, 0.754]
           Ridge (structured + MiniLM)           0.546 [0.457, 0.625]                    0.654 [0.528, 0.748]
Chance level is 0.5.


In [32]:
import os
try:
    import tensorflow as tf          # preferred: the course uses TensorFlow/Keras
    import keras
    from keras import layers
except ModuleNotFoundError:
    os.environ["KERAS_BACKEND"] = "torch"   # fallback: Keras 3 on top of PyTorch (must be set before importing keras)
    import keras
    from keras import layers
import numpy as np
from sklearn.model_selection import GroupShuffleSplit

# ---- Keras neural networks (course lectures 8-10): bag-of-words (TF-IDF) network and a MiniLM-embedding network.
keras.utils.set_random_seed(42)
print("Keras", keras.__version__, "| backend:", keras.backend.backend())

# Early-stopping holdout: 10% of the training split, grouped by product family (the validation split is NOT used for stopping)
fit_idx, es_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=0).split(np.zeros(len(tr)), groups=groups_tr))

# Inputs: leaf dummies + standardized numeric features + a text block
tfidf5 = TfidfVectorizer(min_df=3, ngram_range=(1, 2), max_features=5000, sublinear_tf=True).fit(txt(tr))
B_tr = tfidf5.transform(txt(tr)).toarray().astype("float32")
B_va = tfidf5.transform(txt(va)).toarray().astype("float32")
base_tr = np.hstack([leaf_dummies(tr), sc.transform(num(tr))]).astype("float32")
base_va = np.hstack([leaf_dummies(va), sc.transform(num(va))]).astype("float32")

E_mu, E_sd = E_tr.mean(0), E_tr.std(0) + 1e-6         # standardize the embedding block with training statistics
inputs = {
    "NN (structured + TF-IDF 5k)": (np.hstack([base_tr, B_tr]), np.hstack([base_va, B_va])),
    "NN (structured + MiniLM)":    (np.hstack([base_tr, (E_tr - E_mu) / E_sd]).astype("float32"),
                                    np.hstack([base_va, (E_va - E_mu) / E_sd]).astype("float32")),
}

def build_nn(n_in, units, dropout=0.3, l2=1e-4, lr=1e-3):
    inp = keras.Input(shape=(n_in,))
    h = inp
    for u in units:
        h = layers.Dense(u, activation="relu", kernel_regularizer=keras.regularizers.l2(l2))(h)
        h = layers.Dropout(dropout)(h)
    out = layers.Dense(1, activation="linear")(h)
    m = keras.Model(inp, out)
    m.compile(optimizer=keras.optimizers.Adam(lr), loss="mse")
    return m

for name, (A_tr, A_va) in inputs.items():
    best = None
    for units in [(64,), (128, 64)]:
        t0 = time.time()
        m = build_nn(A_tr.shape[1], units)
        hist = m.fit(A_tr[fit_idx], y_tr[fit_idx], validation_data=(A_tr[es_idx], y_tr[es_idx]),
                     epochs=200, batch_size=64, verbose=0,
                     callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)])
        vl, ep = min(hist.history["val_loss"]), int(np.argmin(hist.history["val_loss"])) + 1
        print(f"{name} units={units}: holdout loss {vl:.4f} at epoch {ep} ({time.time() - t0:.0f}s)")
        if best is None or vl < best[0]:
            best = (vl, units, ep, m)
    vl, units, ep, m = best
    add_result(name, m.predict(A_tr, verbose=0).ravel(), m.predict(A_va, verbose=0).ravel(), f"units={units}, epoch={ep}")

keys = ["Ridge (structured + TF-IDF)", "NN (structured + TF-IDF 5k)", "NN (structured + MiniLM)"]
print(pd.DataFrame([r for r in results if r["model"] in keys]).round(3).to_string(index=False))

Keras 3.15.1 | backend: torch
NN (structured + TF-IDF 5k) units=(64,): holdout loss 0.1873 at epoch 103 (136s)
NN (structured + TF-IDF 5k) units=(128, 64): holdout loss 0.1918 at epoch 81 (145s)
NN (structured + MiniLM) units=(64,): holdout loss 0.1850 at epoch 50 (56s)
NN (structured + MiniLM) units=(128, 64): holdout loss 0.1891 at epoch 58 (103s)
                      model                  tuned  train R2  val OSR2  val MAE  val RMSE   M-R         90% CI  top1
Ridge (structured + TF-IDF)                alpha=1     0.780     0.431    0.281     0.379 0.177 [0.140, 0.213] 0.473
NN (structured + TF-IDF 5k) units=(64,), epoch=103     0.841     0.368    0.294     0.399 0.128 [0.089, 0.169] 0.399
   NN (structured + MiniLM)  units=(64,), epoch=50     0.636     0.347    0.304     0.405 0.171 [0.136, 0.209] 0.459


In [33]:
# ---- FINAL TEST EVALUATION. Run this ONCE: refit on train+val with the settings chosen on validation, then score the untouched test split.
# Do not change models, hyperparameters or set-building rules after seeing these numbers.
from sklearn.preprocessing import OneHotEncoder

tv, te = T[T["split"].isin(["train", "val"])], T[T["split"] == "test"]
y_tv, y_te = tv["mean"].values, te["mean"].values
ybar_tv = y_tv.mean()

# All preprocessing is refit on train+val only
tfidf_f = TfidfVectorizer(min_df=3, ngram_range=(1, 2), max_features=50000, sublinear_tf=True).fit(txt(tv))
ohe_f = OneHotEncoder(handle_unknown="ignore").fit(tv[["cat_leaf"]].fillna("NA"))
med_f = tv["price"].median()
def num_f(d):
    return np.c_[np.log1p(d["price"].fillna(med_f)), d["price"].isna().astype(float),
                 np.log1p(d["title"].str.len()), np.log1p(d["features"].str.len()), np.log1p(d["description"].str.len())]
sc_f = StandardScaler().fit(num_f(tv))
def struct_f(d):
    return hstack([ohe_f.transform(d[["cat_leaf"]].fillna("NA")), csr_matrix(sc_f.transform(num_f(d)))]).tocsr()

Xs_tv, Xs_te = struct_f(tv), struct_f(te)
Xst_tv = hstack([Xs_tv, tfidf_f.transform(txt(tv))]).tocsr()
Xst_te = hstack([Xs_te, tfidf_f.transform(txt(te))]).tocsr()

# Models (settings fixed from the validation stage)
lm_f = tv.groupby("cat_leaf")["mean"].mean()
models = {
    "Baseline: train+val mean": (np.full(len(tv), ybar_tv), np.full(len(te), ybar_tv)),
    "Baseline: leaf mean": (tv["cat_leaf"].map(lm_f).fillna(ybar_tv).values, te["cat_leaf"].map(lm_f).fillna(ybar_tv).values),
}
rb = Ridge(alpha=3).fit(Xs_tv, y_tv)                       # baseline B: structured ridge (no text)
models["Baseline B: Ridge (structured)"] = (rb.predict(Xs_tv), rb.predict(Xs_te))
rp = Ridge(alpha=1).fit(Xst_tv, y_tv)                      # PRIMARY model: Ridge (structured + TF-IDF)
models["PRIMARY: Ridge (structured + TF-IDF)"] = (rp.predict(Xst_tv), rp.predict(Xst_te))

# Test sets: built inside the test split only, same rules as before (category, pack size, price; no ratings used)
ok_te = te[te["set_eligible"] & ~te["cat_leaf"].fillna("").str.contains(MANUAL_EXCLUDE, case=False, regex=True)]
sets_te = []
for (_, _), g_ in ok_te.groupby(["cat_leaf", "pack"]):
    sets_te += make_sets(g_)
pos_te = {ix: k for k, ix in enumerate(te.index)}
sets_pos_te = np.array([[pos_te[i] for i in s] for s in sets_te])
truth_te = y_te[sets_pos_te]
R_te, O_te = truth_te.mean(1), truth_te.max(1)
tie_te = np.random.default_rng(7).normal(0, 1e-9, len(te))
print(f"Test sets: {len(sets_te)} | headroom O-R: {np.mean(O_te - R_te):.3f} stars")

def cluster_boot_mean(vals, groups, B=2000, seed=0):
    d_ = pd.DataFrame({"v": vals, "g": groups}).groupby("g")["v"].agg(["sum", "count"])
    s_, c_ = d_["sum"].values, d_["count"].values
    idx = np.random.default_rng(seed).integers(0, len(d_), (B, len(d_)))
    return np.percentile(s_[idx].sum(1) / c_[idx].sum(1), [5, 95])

rows_f, picked = [], {}
for name, (p_tv, p_te) in models.items():
    j = (p_te + tie_te)[sets_pos_te].argmax(1)
    M = truth_te[np.arange(len(j)), j]
    picked[name] = M
    lo, hi = boot_ci(M - R_te)
    rows_f.append({"model": name, "train+val R2": osr2(y_tv, p_tv, ybar_tv),
                   "test OSR2": osr2(y_te, p_te, ybar_tv), "test MAE": mean_absolute_error(y_te, p_te),
                   "test RMSE": mean_squared_error(y_te, p_te) ** 0.5,
                   "M-R": (M - R_te).mean(), "90% CI": f"[{lo:.3f}, {hi:.3f}]",
                   "top1": (j == truth_te.argmax(1)).mean()})
print(pd.DataFrame(rows_f).round(3).to_string(index=False))

# Primary model vs baseline B: selection gain G_B (per set) and the change in absolute error (clustered by product family)
P, Bn = "PRIMARY: Ridge (structured + TF-IDF)", "Baseline B: Ridge (structured)"
gb = picked[P] - picked[Bn]
lo, hi = boot_ci(gb)
print(f"\nG_B = M - B (mean true rating of primary pick minus baseline pick): {gb.mean():.3f} stars, 90% CI [{lo:.3f}, {hi:.3f}]")
dmae = np.abs(y_te - models[P][1]) - np.abs(y_te - models[Bn][1])
lo, hi = cluster_boot_mean(dmae, te["family"].values)
print(f"Delta MAE (primary - baseline B): {dmae.mean():.3f}, 90% CI [{lo:.3f}, {hi:.3f}]  (negative = primary is better)")

Test sets: 121 | headroom O-R: 0.344 stars
                               model  train+val R2  test OSR2  test MAE  test RMSE    M-R          90% CI  top1
            Baseline: train+val mean         0.000      0.000     0.386      0.499 -0.018 [-0.077, 0.037] 0.264
                 Baseline: leaf mean         0.301      0.237     0.325      0.436 -0.018 [-0.077, 0.037] 0.264
      Baseline B: Ridge (structured)         0.298      0.263     0.322      0.429  0.007 [-0.042, 0.054] 0.264
PRIMARY: Ridge (structured + TF-IDF)         0.774      0.437     0.278      0.375  0.164  [0.114, 0.212] 0.488

G_B = M - B (mean true rating of primary pick minus baseline pick): 0.157 stars, 90% CI [0.089, 0.228]
Delta MAE (primary - baseline B): -0.044, 90% CI [-0.049, -0.040]  (negative = primary is better)


In [34]:
# ---- Confidence routing (course lecture 11): auto-select when the model is confident, send the rest to a human reviewer.
# Confidence of a set = predicted score gap between the model's best and second-best product.
# The thresholds are chosen on the VALIDATION sets (train-only model); they are then applied, unchanged, to the TEST sets.
def set_margins(pred, sets_pos_, tie_):
    srt = np.sort((pred + tie_)[sets_pos_], axis=1)
    return srt[:, -1] - srt[:, -2]

rp_tr = Ridge(alpha=1).fit(Xst_tr, y_tr)
m_va = set_margins(rp_tr.predict(Xst_va), sets_pos, tie)
QS = [0.0, 0.2, 0.4, 0.5, 0.6, 0.8]
taus = np.quantile(m_va, QS)                       # QS = share of validation sets that gets deferred to a human

p_te_prim = models[P][1]
m_te = set_margins(p_te_prim, sets_pos_te, tie_te)
j_te = (p_te_prim + tie_te)[sets_pos_te].argmax(1)
gain_te = picked[P] - R_te                          # per-set gain of the model's pick over a random pick (stars)
hit_te = (j_te == truth_te.argmax(1))

rows_r = []
for q, tau in zip(QS, taus):
    auto = m_te >= tau
    lo, hi = boot_ci(gain_te[auto]) if auto.sum() > 1 else (np.nan, np.nan)
    rows_r.append({"share deferred (val quantile)": q, "margin threshold": tau,
                   "share auto (test)": auto.mean(),
                   "gain on auto sets": gain_te[auto].mean(), "90% CI": f"[{lo:.3f}, {hi:.3f}]",
                   "top1 auto": hit_te[auto].mean(),
                   "gain on deferred": gain_te[~auto].mean() if (~auto).any() else np.nan,
                   "top1 deferred": hit_te[~auto].mean() if (~auto).any() else np.nan})
print("Does confidence (score margin) tell easy sets from hard sets? Test split, thresholds from validation.")
print(pd.DataFrame(rows_r).round(3).to_string(index=False))

# Stylized scenarios: ASSUMPTIONS, not results. h = average gain (stars over a random pick) of a human reviewer;
# the human-vs-model comparison is exactly what the behavioral study in HBSDOC 4435 is designed to measure.
print(f"\nAll sets handled by the model: {gain_te.mean():.3f} stars per set; headroom O-R = {np.mean(O_te - R_te):.3f}")
scen = []
for h in [0.10, 0.20, 0.30]:
    for q in [0.4, 0.5, 0.6]:
        tau = np.quantile(m_va, q)
        auto = m_te >= tau
        routed = auto.mean() * gain_te[auto].mean() + (1 - auto.mean()) * h
        scen.append({"assumed human gain h": h, "share deferred (val quantile)": q,
                     "all-human": h, "all-model": gain_te.mean(), "routed (stars/set)": routed,
                     "human reviews per 100 sets (routed)": 100 * (1 - auto.mean()), "(all-human)": 100})
print(pd.DataFrame(scen).round(3).to_string(index=False))

Does confidence (score margin) tell easy sets from hard sets? Test split, thresholds from validation.
 share deferred (val quantile)  margin threshold  share auto (test)  gain on auto sets         90% CI  top1 auto  gain on deferred  top1 deferred
                           0.0             0.000              1.000              0.164 [0.114, 0.212]      0.488               NaN            NaN
                           0.2             0.024              0.818              0.186 [0.131, 0.245]      0.515             0.064          0.364
                           0.4             0.074              0.603              0.216 [0.148, 0.286]      0.589             0.084          0.333
                           0.5             0.107              0.430              0.206 [0.110, 0.296]      0.596             0.132          0.406
                           0.6             0.140              0.339              0.283 [0.190, 0.369]      0.659             0.103          0.400
                      

In [35]:
# ---- Figures for the deck and README (saved to figures/). Colors follow the course notebooks' colorblind-safe palette.
import os, re
import matplotlib.pyplot as plt

os.makedirs("figures", exist_ok=True)
ACCENT, GREY, ORANGE, INK = "#0072B2", "#8C8C8C", "#D55E00", "#0B0B0B"
plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False})

def ci_vals(s):
    lo, hi = [float(x) for x in re.findall(r"-?\d+\.\d+", s)[:2]]
    return lo, hi

# Figure 1: validation selection gain by model (M-R with 90% CI)
show = ["Baseline: leaf mean", "Ridge (structured)", "Ridge (structured + MiniLM)", "Gradient boosting (structured + MiniLM)",
        "Random forest (structured + TF-IDF SVD)", "Gradient boosting (structured + TF-IDF SVD)",
        "NN (structured + TF-IDF 5k)", "NN (structured + MiniLM)", "Ridge (structured + TF-IDF)"]
rows1 = [r for name in show for r in results if r["model"] == name]
fig, ax = plt.subplots(figsize=(8, 4.8))
y = np.arange(len(rows1))
vals = np.array([r["M-R"] for r in rows1]); cis = np.array([ci_vals(r["90% CI"]) for r in rows1])
colors = [ACCENT if r["model"] == "Ridge (structured + TF-IDF)" else GREY for r in rows1]
ax.barh(y, vals, xerr=[vals - cis[:, 0], cis[:, 1] - vals], color=colors, ecolor=INK, capsize=3)
ax.set_yticks(y); ax.set_yticklabels([r["model"] for r in rows1]); ax.axvline(0, color=INK, lw=0.8)
ax.set_xlabel("Selection gain over a random pick, stars (validation sets, 90% CI)")
ax.set_title("Which model picks better products?")
fig.tight_layout(); fig.savefig("figures/fig1_validation_models.png", dpi=200); plt.close(fig)

# Figure 2: final test results
names_f = ["Baseline: leaf mean", "Baseline B: Ridge (structured)", "PRIMARY: Ridge (structured + TF-IDF)"]
rows2 = [r for name in names_f for r in rows_f if r["model"] == name]
fig, ax = plt.subplots(figsize=(7.5, 3.6))
y = np.arange(len(rows2))
vals = np.array([r["M-R"] for r in rows2]); cis = np.array([ci_vals(r["90% CI"]) for r in rows2])
ax.barh(y, vals, xerr=[vals - cis[:, 0], cis[:, 1] - vals],
        color=[GREY, GREY, ACCENT], ecolor=INK, capsize=3)
ax.set_yticks(y); ax.set_yticklabels(["Category average", "Structured features only", "Text model (primary)"])
ax.axvline(0, color=INK, lw=0.8)
ax.set_xlabel("Selection gain over a random pick, stars (test sets, 90% CI)")
ax.set_title("Final test: the text model picks better products")
fig.tight_layout(); fig.savefig("figures/fig2_test_results.png", dpi=200); plt.close(fig)

# Figure 3: confidence vs gain
d = pd.DataFrame(rows_r)
d = d[d["share auto (test)"] > 0]
cis = np.array([ci_vals(s) for s in d["90% CI"]])
fig, ax = plt.subplots(figsize=(7, 4.2))
x = d["share auto (test)"].values * 100; v = d["gain on auto sets"].values
ax.errorbar(x, v, yerr=[v - cis[:, 0], cis[:, 1] - v], fmt="o-", color=ACCENT, ecolor=INK, capsize=3)
ax.axhline(gain_te.mean(), color=ORANGE, ls="--", lw=1)
ax.text(x.min(), gain_te.mean() + 0.008, "average over all sets", color=ORANGE, fontsize=9)
ax.set_xlabel("Share of sets handled automatically (most confident first), %")
ax.set_ylabel("Selection gain on those sets, stars")
ax.set_title("The more confident the model, the better its picks")
ax.invert_xaxis()
fig.tight_layout(); fig.savefig("figures/fig3_confidence_vs_gain.png", dpi=200); plt.close(fig)

# Figure 4: terms most associated with lower / higher ratings (within-leaf model; associations, not causal effects)
k = 12
idx_sel = list(order[:k]) + list(order[::-1][:k][::-1])
fig, ax = plt.subplots(figsize=(7.5, 6))
ax.barh(np.arange(len(idx_sel)), coef[idx_sel], color=[ORANGE] * k + [ACCENT] * k)
ax.set_yticks(np.arange(len(idx_sel))); ax.set_yticklabels(names[idx_sel]); ax.axvline(0, color=INK, lw=0.8)
ax.set_xlabel("Ridge coefficient (higher = associated with higher average rating)")
ax.set_title("Terms associated with lower and higher ratings")
fig.tight_layout(); fig.savefig("figures/fig4_terms.png", dpi=200); plt.close(fig)
print("Saved:", sorted(os.listdir("figures")))

Saved: ['fig1_validation_models.png', 'fig2_test_results.png', 'fig3_confidence_vs_gain.png', 'fig4_terms.png']


In [36]:
# ---- Figures v2 (deck-ready): readable labels, sorted bars, sample sizes shown, notes on interpretation. Saved to figures/.
import os, re
import matplotlib.pyplot as plt

os.makedirs("figures", exist_ok=True)
ACCENT, GREY, ORANGE, INK = "#0072B2", "#8C8C8C", "#D55E00", "#0B0B0B"
plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False})

def ci_vals(s):
    lo, hi = [float(x) for x in re.findall(r"-?\d+\.\d+", s)[:2]]
    return lo, hi

# Figure 1: validation selection gain by model, sorted, plain-language labels
nice = {"Baseline: leaf mean": "Category average (baseline)",
        "Ridge (structured)": "Ridge, no text (baseline B)",
        "Ridge (structured + MiniLM)": "Ridge + sentence embeddings",
        "Gradient boosting (structured + MiniLM)": "Gradient boosting + sentence embeddings",
        "Random forest (structured + TF-IDF SVD)": "Random forest + text",
        "Gradient boosting (structured + TF-IDF SVD)": "Gradient boosting + text",
        "NN (structured + TF-IDF 5k)": "Neural net + text (TF-IDF)",
        "NN (structured + MiniLM)": "Neural net + sentence embeddings",
        "Ridge (structured + TF-IDF)": "Ridge + text (primary model)"}
rows1 = sorted([r for name in nice for r in results if r["model"] == name], key=lambda r: r["M-R"])
fig, ax = plt.subplots(figsize=(8.5, 4.8))
y = np.arange(len(rows1))
vals = np.array([r["M-R"] for r in rows1]); cis = np.array([ci_vals(r["90% CI"]) for r in rows1])
colors = [ACCENT if r["model"] == "Ridge (structured + TF-IDF)" else GREY for r in rows1]
ax.barh(y, vals, xerr=[vals - cis[:, 0], cis[:, 1] - vals], color=colors, ecolor=INK, capsize=3)
ax.set_yticks(y); ax.set_yticklabels([nice[r["model"]] for r in rows1]); ax.axvline(0, color=INK, lw=0.8)
ax.set_xlabel("Gain over a random pick (stars), 90% interval")
ax.set_title("Validation: which model picks better products?")
fig.tight_layout(); fig.savefig("figures/fig1_validation_models.png", dpi=200, bbox_inches="tight"); plt.close(fig)

# Figure 2: final test results, with values and number of sets
names_f = ["Baseline: leaf mean", "Baseline B: Ridge (structured)", "PRIMARY: Ridge (structured + TF-IDF)"]
rows2 = [r for name in names_f for r in rows_f if r["model"] == name]
fig, ax = plt.subplots(figsize=(7.5, 3.6))
y = np.arange(len(rows2))
vals = np.array([r["M-R"] for r in rows2]); cis = np.array([ci_vals(r["90% CI"]) for r in rows2])
ax.barh(y, vals, xerr=[vals - cis[:, 0], cis[:, 1] - vals], color=[GREY, GREY, ACCENT], ecolor=INK, capsize=3)
for yi, v, hi_ in zip(y, vals, cis[:, 1]):
    ax.text(hi_ + 0.006, yi, f"{v:.3f}", va="center", fontsize=10)
ax.set_yticks(y); ax.set_yticklabels(["Category average", "Structured features only", "Text model (primary)"])
ax.axvline(0, color=INK, lw=0.8)
ax.set_xlabel(f"Gain over a random pick (stars), 90% interval; {len(sets_te)} test sets")
ax.set_title("Final test: the text model picks better products")
fig.tight_layout(); fig.savefig("figures/fig2_test_results.png", dpi=200, bbox_inches="tight"); plt.close(fig)

# Figure 3: confidence vs gain, with the number of sets behind each point
d = pd.DataFrame(rows_r)
d = d[d["share auto (test)"] > 0]
cis = np.array([ci_vals(s) for s in d["90% CI"]])
fig, ax = plt.subplots(figsize=(7.5, 4.4))
x = d["share auto (test)"].values * 100; v = d["gain on auto sets"].values
ax.errorbar(x, v, yerr=[v - cis[:, 0], cis[:, 1] - v], fmt="o-", color=ACCENT, ecolor=INK, capsize=3)
for xi, hi_, sh in zip(x, cis[:, 1], d["share auto (test)"].values):
    ax.annotate(f"n={int(round(sh * len(sets_te)))}", (xi, hi_), textcoords="offset points", xytext=(0, 5), ha="center", fontsize=8)
ax.axhline(gain_te.mean(), color=ORANGE, ls="--", lw=1)
ax.text(np.median(x), gain_te.mean() - 0.035, "average over all sets", color=ORANGE, fontsize=9, ha="center")
ax.set_ylim(0, cis[:, 1].max() + 0.06)
ax.set_xlabel("Share of test sets handled automatically (most confident first), %")
ax.set_ylabel("Gain over a random pick (stars)")
ax.set_title("Picks tend to be better where the model is more confident")
ax.invert_xaxis()
fig.tight_layout(); fig.savefig("figures/fig3_confidence_vs_gain.png", dpi=200, bbox_inches="tight"); plt.close(fig)

# Figure 4: terms most associated with lower / higher ratings (purely numeric tokens hidden for readability)
k = 10
keep_tok = np.array([not re.fullmatch(r"[\d\s]+", t) for t in names])
order_f = [i for i in order if keep_tok[i]]
idx_sel = list(order_f[:k]) + list(order_f[::-1][:k][::-1])
fig, ax = plt.subplots(figsize=(7.5, 5.6))
ax.barh(np.arange(len(idx_sel)), coef[idx_sel], color=[ORANGE] * k + [ACCENT] * k)
ax.set_yticks(np.arange(len(idx_sel))); ax.set_yticklabels(names[idx_sel]); ax.axvline(0, color=INK, lw=0.8)
ax.set_xlabel("Model coefficient (positive = associated with higher average rating)")
ax.set_title("Terms associated with lower and higher ratings")
fig.text(0.01, -0.02, "Associations, not causal effects. Many terms are brand names or product types, not quality.\nPurely numeric terms are hidden for readability.", fontsize=8, va="top")
fig.tight_layout(); fig.savefig("figures/fig4_terms.png", dpi=200, bbox_inches="tight"); plt.close(fig)
print("Saved:", sorted(os.listdir("figures")))

Saved: ['fig1_validation_models.png', 'fig2_test_results.png', 'fig3_confidence_vs_gain.png', 'fig4_terms.png']
